# WRDS Data Query - IvyDB OptionMetrics

This notebook queries S&P 100 equity options data from WRDS:
- **Options**: OptionMetrics IvyDB `opprcd` tables (2011-2023)
- **Equity Prices**: CRSP Daily Stock File (`dsf`)
- **Corporate Actions**: CRSP distributions (`dsedist`) for split adjustments

**Filters applied in post-processing:**
- **Standard contracts only**: ss_flag == '0' AND contract_size == 100
  - This excludes adjusted/non-standard contracts created after stock splits
- Exclude zero open interest
- Moneyness (S/K for calls, K/S for puts) between 0.9 and 1.1

**Memory Management:**
- Options queried and saved year-by-year (avoids loading ~30GB at once)
- Filtered results also stored year-by-year
- Combined file created only if total size < 4GB

## 1. Setup

In [ ]:
import wrds
import pandas as pd
import numpy as np
from pathlib import Path
import os
from tqdm import tqdm

# Configuration
START_YEAR = 2011
END_YEAR = 2023
OUTPUT_DIR = Path('raw_data')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Query range: {START_YEAR}-{END_YEAR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load security IDs
with open('secids.txt', 'r') as f:
    secids = [int(line.strip()) for line in f if line.strip()]

print(f"Loaded {len(secids)} security IDs")
print(f"First 10: {secids[:10]}")

In [ ]:
# Connect to WRDS
# Note: You need WRDS credentials. First time will prompt for username/password.
db = wrds.Connection()
print("Connected to WRDS")

## 2. Query Options Data (opprcd tables)

The OptionMetrics option prices are stored in year-sharded tables: `opprcd2011`, `opprcd2012`, etc.

Fields:
- `secid`: Security identifier
- `date`: Quote date
- `exdate`: Expiration date
- `cp_flag`: Call (C) or Put (P)
- `strike_price`: Strike price (in dollars, needs division by 1000)
- `best_bid`, `best_offer`: Bid-ask quotes
- `impl_volatility`: Implied volatility
- `delta`: Option delta
- `open_interest`: Open interest
- `volume`: Trading volume

In [ ]:
def query_options_year(db, year: int, secids: list) -> pd.DataFrame:
    """
    Query options data for a single year.
    
    Args:
        db: WRDS connection
        year: Year to query
        secids: List of security IDs
    
    Returns:
        DataFrame with options data
    """
    table_name = f'optionm.opprcd{year}'
    secid_str = ','.join(map(str, secids))
    
    query = f"""
    SELECT 
        secid,
        date,
        exdate,
        cp_flag,
        strike_price,
        best_bid,
        best_offer,
        impl_volatility,
        delta,
        open_interest,
        volume,
        ss_flag,
        contract_size
    FROM {table_name}
    WHERE secid IN ({secid_str})
      AND open_interest > 0
    """
    
    df = db.raw_sql(query)
    
    # Optimize dtypes to reduce memory (~40% reduction)
    if len(df) > 0:
        df['secid'] = df['secid'].astype('int32')
        df['strike_price'] = df['strike_price'].astype('float32')
        df['best_bid'] = df['best_bid'].astype('float32')
        df['best_offer'] = df['best_offer'].astype('float32')
        df['impl_volatility'] = df['impl_volatility'].astype('float32')
        df['delta'] = df['delta'].astype('float32')
        df['open_interest'] = df['open_interest'].astype('int32')
        df['volume'] = df['volume'].astype('int32')
        df['cp_flag'] = df['cp_flag'].astype('category')
        df['ss_flag'] = df['ss_flag'].astype('category')
        df['contract_size'] = df['contract_size'].astype('int32')
    
    return df


def get_options_count(db, year: int, secids: list) -> int:
    """Get total record count for a year (to determine if chunking needed)."""
    table_name = f'optionm.opprcd{year}'
    secid_str = ','.join(map(str, secids))
    
    query = f"""
    SELECT COUNT(*) as cnt
    FROM {table_name}
    WHERE secid IN ({secid_str})
      AND open_interest > 0
    """
    result = db.raw_sql(query)
    return int(result['cnt'].iloc[0])


def query_options_year_chunked(db, year: int, secids: list, chunk_size: int = 5_000_000) -> pd.DataFrame:
    """
    Query options data for a single year in chunks to handle large datasets.
    
    Args:
        db: WRDS connection
        year: Year to query
        secids: List of security IDs
        chunk_size: Maximum records per chunk (default 5 million)
    
    Returns:
        DataFrame with options data
    """
    table_name = f'optionm.opprcd{year}'
    secid_str = ','.join(map(str, secids))
    
    chunks = []
    offset = 0
    
    while True:
        query = f"""
        SELECT 
            secid,
            date,
            exdate,
            cp_flag,
            strike_price,
            best_bid,
            best_offer,
            impl_volatility,
            delta,
            open_interest,
            volume,
            ss_flag,
            contract_size
        FROM {table_name}
        WHERE secid IN ({secid_str})
          AND open_interest > 0
        ORDER BY secid, date, exdate, cp_flag, strike_price
        LIMIT {chunk_size} OFFSET {offset}
        """
        
        chunk_df = db.raw_sql(query)
        
        if len(chunk_df) == 0:
            break
        
        # Optimize dtypes to reduce memory
        chunk_df['secid'] = chunk_df['secid'].astype('int32')
        chunk_df['strike_price'] = chunk_df['strike_price'].astype('float32')
        chunk_df['best_bid'] = chunk_df['best_bid'].astype('float32')
        chunk_df['best_offer'] = chunk_df['best_offer'].astype('float32')
        chunk_df['impl_volatility'] = chunk_df['impl_volatility'].astype('float32')
        chunk_df['delta'] = chunk_df['delta'].astype('float32')
        chunk_df['open_interest'] = chunk_df['open_interest'].astype('int32')
        chunk_df['volume'] = chunk_df['volume'].astype('int32')
        chunk_df['cp_flag'] = chunk_df['cp_flag'].astype('category')
        chunk_df['ss_flag'] = chunk_df['ss_flag'].astype('category')
        chunk_df['contract_size'] = chunk_df['contract_size'].astype('int32')
        
        chunks.append(chunk_df)
        print(f"    Chunk {len(chunks)}: {len(chunk_df):,} records (offset {offset:,})")
        
        offset += chunk_size
        
        # If we got fewer records than chunk_size, we've reached the end
        if len(chunk_df) < chunk_size:
            break
    
    if chunks:
        return pd.concat(chunks, ignore_index=True)
    else:
        return pd.DataFrame()

In [ ]:
# Query options data year by year - SAVE EACH YEAR IMMEDIATELY TO DISK
# This prevents loading all ~30GB into memory at once
# For years with >5M records, use chunked querying to avoid memory crashes

options_dir = OUTPUT_DIR / 'options_by_year'
options_dir.mkdir(exist_ok=True)

CHUNK_THRESHOLD = 5_000_000  # Use chunked query if year has more than this many records
CHUNK_SIZE = 5_000_000       # Records per chunk

total_records = 0

In [ ]:
for year in tqdm(range(START_YEAR, END_YEAR + 1), desc="Querying options"):
    output_file = options_dir / f'options_{year}.parquet'
    
    # Skip if already downloaded
    if output_file.exists():
        existing_df = pd.read_parquet(output_file)
        n_records = len(existing_df)
        total_records += n_records
        print(f"  {year}: {n_records:,} records (cached)")
        del existing_df  # Free memory
        continue
    
    try:
        # First, check how many records this year has
        record_count = get_options_count(db, year, secids)
        print(f"  {year}: {record_count:,} total records to fetch")
        
        if record_count > CHUNK_THRESHOLD:
            # Use chunked querying for large years
            print(f"  {year}: Using chunked query (>{CHUNK_THRESHOLD:,} records)")
            df = query_options_year_chunked(db, year, secids, chunk_size=CHUNK_SIZE)
        else:
            # Use regular query for smaller years
            df = query_options_year(db, year, secids)
        
        if len(df) > 0:
            # Convert strike price immediately
            df['strike_price'] = df['strike_price'] / 1000
            df['date'] = pd.to_datetime(df['date'])
            df['exdate'] = pd.to_datetime(df['exdate'])
            
            # Save immediately to disk
            df.to_parquet(output_file, index=False)
            n_records = len(df)
            total_records += n_records
            print(f"  {year}: {n_records:,} records (saved)")
            
            # Free memory
            del df
        else:
            print(f"  {year}: No data")
    except Exception as e:
        print(f"  {year}: Error - {e}")

print(f"\nTotal options records across all years: {total_records:,}")
print(f"Files saved to: {options_dir}/")

    Chunk 1: 5,000,000 records (offset 0)
    Chunk 2: 4,784,229 records (offset 5,000,000)


Querying options:  31%|███       | 4/13 [45:30<1:56:15, 775.03s/it]

  2014: 9,784,229 records (saved)
  2015: 10,670,629 total records to fetch
  2015: Using chunked query (>5,000,000 records)
    Chunk 1: 5,000,000 records (offset 0)
    Chunk 2: 5,000,000 records (offset 5,000,000)
    Chunk 3: 670,629 records (offset 10,000,000)


Querying options:  38%|███▊      | 5/13 [1:05:35<2:04:01, 930.13s/it]

  2015: 10,670,629 records (saved)
  2016: 10,755,837 total records to fetch
  2016: Using chunked query (>5,000,000 records)
    Chunk 1: 5,000,000 records (offset 0)
    Chunk 2: 5,000,000 records (offset 5,000,000)
    Chunk 3: 755,837 records (offset 10,000,000)


...

  2020: 18,368,041 records (saved)
  2021: 18,597,364 total records to fetch
  2021: Using chunked query (>5,000,000 records)
    Chunk 1: 5,000,000 records (offset 0)
    Chunk 2: 5,000,000 records (offset 5,000,000)
    Chunk 3: 5,000,000 records (offset 10,000,000)
    Chunk 4: 3,597,364 records (offset 15,000,000)


Querying options:  85%|████████▍ | 11/13 [3:39:33<53:42, 1611.47s/it]  

  2021: 18,597,364 records (saved)
  2022: 18,569,604 total records to fetch
  2022: Using chunked query (>5,000,000 records)
    Chunk 1: 5,000,000 records (offset 0)
    Chunk 2: 5,000,000 records (offset 5,000,000)
    Chunk 3: 5,000,000 records (offset 10,000,000)
    Chunk 4: 3,569,604 records (offset 15,000,000)


Querying options:  92%|█████████▏| 12/13 [4:24:45<32:26, 1946.26s/it]

  2022: 18,569,604 records (saved)
  2023: 17,470,711 total records to fetch
  2023: Using chunked query (>5,000,000 records)
    Chunk 1: 5,000,000 records (offset 0)


In [ ]:
# List downloaded files
print("Downloaded option files:")
for f in sorted(options_dir.glob('*.parquet')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
# Options data is already saved year-by-year in options_by_year/
# No need to combine into single file (would be ~30GB)

## 3. Query Underlying Equity Prices (CRSP)

Get daily equity prices from CRSP to compute moneyness.

We need to join OptionMetrics secid to CRSP permno via the `optionm.securd` linking table.

In [ ]:
def query_equity_prices(db, secids: list, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Query equity prices from CRSP, linked via historical CUSIP matching.
    
    Uses WRDS best practice: link OptionMetrics SECID to CRSP PERMNO by matching
    historical CUSIP from secnmd (OptionMetrics) to NCUSIP from stocknames (CRSP).
    
    Args:
        db: WRDS connection
        secids: List of OptionMetrics security IDs
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
    
    Returns:
        DataFrame with secid, date, price, cumulative adjustment factor
    """
    secid_str = ','.join(map(str, secids))
    
    query = f"""
    SELECT DISTINCT
        a.secid,
        c.date,
        c.prc,
        c.cfacpr
    FROM optionm.secnmd a
    JOIN crsp.stocknames b 
        ON a.cusip = b.ncusip
    JOIN crsp.dsf c 
        ON b.permno = c.permno
        AND c.date BETWEEN b.namedt AND COALESCE(b.nameenddt, c.date)
    WHERE a.secid IN ({secid_str})
      AND c.date BETWEEN '{start_date}' AND '{end_date}'
    ORDER BY a.secid, c.date
    """
    
    df = db.raw_sql(query)
    return df

In [ ]:
# Query equity prices
print("Querying equity prices from CRSP...")
equity_df = query_equity_prices(
    db, 
    secids, 
    f'{START_YEAR}-01-01', 
    f'{END_YEAR}-12-31'
)

print(f"Total equity price records: {len(equity_df):,}")

# Convert date and handle negative prices (CRSP convention for bid-ask average)
equity_df['date'] = pd.to_datetime(equity_df['date'])
equity_df['prc'] = equity_df['prc'].abs()  # CRSP uses negative for bid-ask avg

equity_df.head()

In [ ]:
# Save equity prices
equity_df.to_parquet(OUTPUT_DIR / 'equity_prices_raw.parquet', index=False)
print(f"Saved equity prices to {OUTPUT_DIR / 'equity_prices_raw.parquet'}")

In [ ]:
# Verify options files are valid

import pandas as pd
from pathlib import Path

data_dir = Path("/home/adam/new4YP/raw_data/options_by_year")  # change
files = sorted(data_dir.glob("*.parquet"))

def inspect_file(fp, n=5):
    df = pd.read_parquet(fp)
    print("\n===", fp.name, "===")
    print("rows:", len(df), "cols:", df.shape[1])
    print("columns:", df.columns.tolist())
    print("\ndtypes:\n", df.dtypes.sort_index())
    print("\nhead:\n", df.head(n))
    print("\nmissing % (top 20):\n", (df.isna().mean().sort_values(ascending=False).head(20)*100).round(2))
    return df

# inspect one year (pick a representative file)
df0 = inspect_file(files[0])

## 4. Query Stock Splits / Corporate Actions (CRSP)

Get distribution events to adjust strike prices for splits.

In [ ]:
def query_distributions(db, secids: list, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Query stock splits and distributions from CRSP.
    
    Args:
        db: WRDS connection
        secids: List of OptionMetrics security IDs
        start_date: Start date
        end_date: End date
    
    Returns:
        DataFrame with split/distribution events
    """
    secid_str = ','.join(map(str, secids))
    
    query = f"""
        SELECT DISTINCT
            a.secid,
            c.permno,
            c.exdt,
            c.distcd,
            c.facpr,
            c.facshr
        FROM optionm.secnmd a
        JOIN crsp.stocknames b
            ON a.cusip = b.ncusip
        JOIN crsp.dsedist c
            ON b.permno = c.permno
            AND c.exdt BETWEEN b.namedt AND COALESCE(b.nameenddt, c.exdt)
        WHERE a.secid IN ({secid_str})
            AND c.exdt BETWEEN '{start_date}' AND '{end_date}'
            AND (c.facpr IS NOT NULL OR c.facshr IS NOT NULL)
        ORDER BY a.secid, c.exdt
    """
    
    df = db.raw_sql(query)
    return df

In [ ]:
# Query distributions
print("Querying corporate actions from CRSP...")
distributions_df = query_distributions(
    db,
    secids,
    f'{START_YEAR}-01-01',
    f'{END_YEAR}-12-31'
)

print(f"Total distribution records: {len(distributions_df):,}")

# Convert date
if len(distributions_df) > 0:
    distributions_df['exdt'] = pd.to_datetime(distributions_df['exdt'])

    # Filter to stock splits only (distcd 5xxx)
    splits_df = distributions_df[
        (distributions_df['distcd'] >= 5000) &
        (distributions_df['distcd'] < 6000)
    ].copy()
    print(f"Stock splits: {len(splits_df)}")
else:
    splits_df = pd.DataFrame()
    print("No distribution data found")

distributions_df.head(10)

In [ ]:
splits_df[["facpr","facshr"]].describe()
splits_df.sort_values(["secid","exdt"]).head(30)

In [ ]:
# Save distributions
distributions_df.to_parquet(OUTPUT_DIR / 'distributions_raw.parquet', index=False)
print(f"Saved distributions to {OUTPUT_DIR / 'distributions_raw.parquet'}")

## 5. Get Security Metadata

Get ticker symbols and company names for the secids.

In [ ]:
def query_security_names(db, secids: list) -> pd.DataFrame:
    """
    Query security names and tickers from OptionMetrics.
    """
    secid_str = ','.join(map(str, secids))
    
    query = f"""
    SELECT DISTINCT
        secid,
        ticker,
        cusip,
        issuer
    FROM optionm.secnmd
    WHERE secid IN ({secid_str})
    """
    
    df = db.raw_sql(query)
    return df

In [ ]:
# Query security names
print("Querying security metadata...")
secnames_df = query_security_names(db, secids)
print(f"Securities found: {len(secnames_df)}")

# Get most recent ticker for each secid
secnames_df = secnames_df.drop_duplicates(subset='secid', keep='last')

print("\nSecurity list:")
print(secnames_df[['secid', 'ticker', 'issuer']].head(20).to_string(index=False))

In [ ]:
# Save security metadata
secnames_df.to_parquet(OUTPUT_DIR / 'security_names.parquet', index=False)
print(f"Saved security names to {OUTPUT_DIR / 'security_names.parquet'}")

In [ ]:
# Close WRDS connection
db.close()
print("\nWRDS connection closed.")

## 6. Post-Processing: Compute Moneyness and Filter

Now we merge options with equity prices and filter by moneyness.

In [ ]:
# Load equity prices (this is manageable in memory - ~300k rows for 87 securities over 13 years)
equity_df = pd.read_parquet(OUTPUT_DIR / 'equity_prices_raw.parquet')
equity_df['date'] = pd.to_datetime(equity_df['date'])
equity_df = equity_df.rename(columns={'prc': 'spot_price'})

print(f"Equity prices loaded: {len(equity_df):,} records")
print(f"Memory usage: {equity_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# Process options YEAR BY YEAR to avoid memory issues
# Merge with equity, compute moneyness, filter, save filtered results

MIN_MONEYNESS = 0.9
MAX_MONEYNESS = 1.1

filtered_dir = OUTPUT_DIR / 'options_filtered_by_year'
filtered_dir.mkdir(exist_ok=True)

In [ ]:
def compute_moneyness_vectorized(df):
    """Vectorized moneyness computation - much faster than apply()"""
    moneyness = np.where(
        df['cp_flag'] == 'C',
        df['spot_price'] / df['strike_price'],  # Calls: S/K
        df['strike_price'] / df['spot_price']   # Puts: K/S
    )
    return moneyness

total_before = 0
total_after = 0
total_nonstandard_removed = 0

for year in tqdm(range(START_YEAR, END_YEAR + 1), desc="Processing"):
    input_file = options_dir / f'options_{year}.parquet'
    output_file = filtered_dir / f'options_filtered_{year}.parquet'
    
    # Skip if already processed
    if output_file.exists():
        filtered_df = pd.read_parquet(output_file)
        total_after += len(filtered_df)
        print(f"  {year}: {len(filtered_df):,} filtered records (cached)")
        del filtered_df
        continue
    
    if not input_file.exists():
        print(f"  {year}: No input file")
        continue
    
    # Load year's options
    options_year = pd.read_parquet(input_file)
    options_year['date'] = pd.to_datetime(options_year['date'])
    total_before += len(options_year)
    
    # Filter to STANDARD contracts only (ss_flag == '0' and contract_size == 100)
    # This removes adjusted/non-standard contracts created after stock splits
    before_std_filter = len(options_year)
    options_year = options_year[
        (options_year['ss_flag'] == '0') & 
        (options_year['contract_size'] == 100)
    ]
    nonstandard_removed = before_std_filter - len(options_year)
    total_nonstandard_removed += nonstandard_removed
    
    if len(options_year) == 0:
        print(f"  {year}: No standard contracts found")
        continue
    
    # Merge with equity prices
    merged = options_year.merge(
        equity_df[['secid', 'date', 'spot_price', 'cfacpr']],
        on=['secid', 'date'],
        how='inner'
    )
    
    if len(merged) == 0:
        print(f"  {year}: No matches after merge")
        del options_year, merged
        continue
    
    # Compute moneyness (vectorized)
    merged['moneyness'] = compute_moneyness_vectorized(merged)
    merged['spot_strike_ratio'] = merged['spot_price'] / merged['strike_price']
    
    # Filter by moneyness
    filtered = merged[
        (merged['moneyness'] >= MIN_MONEYNESS) & 
        (merged['moneyness'] <= MAX_MONEYNESS)
    ].copy()
    
    total_after += len(filtered)
    
    # Save filtered data
    filtered.to_parquet(output_file, index=False)
    print(f"  {year}: {before_std_filter:,} -> {len(options_year):,} (std) -> {len(filtered):,} (moneyness) | "
          f"removed {nonstandard_removed:,} non-std")
    
    # Free memory
    del options_year, merged, filtered

print(f"\n=== Summary ===")
print(f"Total before filter: {total_before:,}")
print(f"Non-standard contracts removed: {total_nonstandard_removed:,}")
print(f"Total after filter: {total_after:,}")
#print(f"Reduction: {100*(1 - total_after/total_before):.1f}%")

In [ ]:
# List filtered files and their sizes
print("Filtered option files:")
total_size = 0
for f in sorted(filtered_dir.glob('*.parquet')):
    size_mb = f.stat().st_size / (1024 * 1024)
    total_size += size_mb
    print(f"  {f.name}: {size_mb:.1f} MB")
print(f"\nTotal filtered data size: {total_size:.1f} MB")

In [ ]:
# Summary statistics (load sample if full data not in memory)
# Load just one year as sample
sample_file = list(filtered_dir.glob('*.parquet'))[0]
sample_df = pd.read_parquet(sample_file)
print(f"(Showing statistics for sample year: {sample_file.name})\n")

print("=== Filtered Dataset Summary ===")
print(f"Date range: {sample_df['date'].min()} to {sample_df['date'].max()}")
print(f"Records in sample: {len(sample_df):,}")
print(f"Unique securities: {sample_df['secid'].nunique()}")
print(f"Unique dates: {sample_df['date'].nunique()}")

print("\nBy option type:")
print(sample_df['cp_flag'].value_counts())

print("\nMoneyness distribution:")
print(sample_df['moneyness'].describe())

# if filtered_df is None:
#     del sample_df

In [ ]:
# # Save combined filtered dataset if it fits in memory
# if filtered_df is not None:
#     filtered_df.to_parquet(OUTPUT_DIR / 'options_filtered_combined.parquet', index=False)
#     print(f"Saved combined filtered options to {OUTPUT_DIR / 'options_filtered_combined.parquet'}")
    
#     # Save sample CSV for inspection
#     filtered_df.head(10000).to_csv(OUTPUT_DIR / 'options_filtered_sample.csv', index=False)
#     print(f"Saved sample to {OUTPUT_DIR / 'options_filtered_sample.csv'}")
# else:
#     print("Data stored in year-by-year parquet files:")
#     print(f"  {filtered_dir}/")

## 7. Data Quality Checks

In [ ]:
# Check coverage by security (aggregate across all years)
print("Counting records per security across all years...")

sec_counts = {}
for f in tqdm(sorted(filtered_dir.glob('*.parquet')), desc="Counting"):
    year_df = pd.read_parquet(f, columns=['secid'])
    for secid, count in year_df['secid'].value_counts().items():
        sec_counts[secid] = sec_counts.get(secid, 0) + count
    del year_df

sec_counts = pd.Series(sec_counts).sort_values(ascending=False)

print(f"\nRecords per security (top 20):")
print(sec_counts.head(20))

print(f"\nMin records: {sec_counts.min():,}")
print(f"Max records: {sec_counts.max():,}")
print(f"Mean records: {sec_counts.mean():,.0f}")
print(f"Total securities: {len(sec_counts)}")

In [ ]:
# Check for missing values (sample one year)
sample_file = list(filtered_dir.glob('*.parquet'))[6]  # Pick middle year
sample_df = pd.read_parquet(sample_file)

print(f"Missing values check (sample: {sample_file.name}):")
print(sample_df.isnull().sum())

del sample_df

In [ ]:
# Check bid-ask spread quality (sample one year)
sample_file = list(filtered_dir.glob('*.parquet'))[6]  # Pick middle year
sample_df = pd.read_parquet(sample_file)

sample_df['spread'] = sample_df['best_offer'] - sample_df['best_bid']
sample_df['mid_price'] = (sample_df['best_bid'] + sample_df['best_offer']) / 2
sample_df['spread_pct'] = sample_df['spread'] / sample_df['mid_price'] * 100

print(f"Bid-ask spread % (sample: {sample_file.name}):")
print(sample_df['spread_pct'].describe())

del sample_df

In [ ]:
# Final summary
print("\n" + "="*60)
print("DATA QUERY COMPLETE")
print("="*60)

print(f"\nOutput directory: {OUTPUT_DIR}/")
print("\nRaw data:")
for f in OUTPUT_DIR.glob('*.parquet'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

print(f"\nOptions by year: {options_dir}/")
total_raw = sum(f.stat().st_size for f in options_dir.glob('*.parquet')) / (1024**3)
print(f"  Total: {total_raw:.2f} GB across {len(list(options_dir.glob('*.parquet')))} files")

print(f"\nFiltered options by year: {filtered_dir}/")
total_filtered = sum(f.stat().st_size for f in filtered_dir.glob('*.parquet')) / (1024**3)
print(f"  Total: {total_filtered:.2f} GB across {len(list(filtered_dir.glob('*.parquet')))} files")

print("\n" + "="*60)
print("Next steps:")
print("  1. Use filtered parquet files for straddle formation")
print("  2. Load year-by-year to avoid memory issues")
print("  3. Or use dask/polars for out-of-core processing")
print("="*60)